# Toxicity Peptide Classification with AAIndex

This notebook follows the same AAIndex-based classification workflow as peptideclassification.ipynb, adapted for separate toxicity train/test files.

In [2]:
%pip install aaindex xgboost shap scikit-learn imbalanced-learn pandas numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'e:\COLLEGE\AMRITA VISHWA VIDYPEETHAM _ NEW\New folder\DD\diffusion_bbbp\ve\Scripts\python.exe -m pip install --upgrade pip' command.


In [3]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from aaindex import aaindex1

from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, average_precision_score
)
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
STANDARD_AA = set('ACDEFGHIKLMNPQRSTVWY')

TRAIN_NEG_PATH = r'E:\\COLLEGE\\AMRITA VISHWA VIDYPEETHAM _ NEW\\New folder\\DD\\diffusion_bbbp\\train_neg.csv'
TRAIN_POS_PATH = r'E:\\COLLEGE\\AMRITA VISHWA VIDYPEETHAM _ NEW\\New folder\\DD\\diffusion_bbbp\\train_pos.csv'
TEST_NEG_PATH  = r'E:\\COLLEGE\\AMRITA VISHWA VIDYPEETHAM _ NEW\\New folder\\DD\\diffusion_bbbp\\test_neg.csv'
TEST_POS_PATH  = r'E:\\COLLEGE\\AMRITA VISHWA VIDYPEETHAM _ NEW\\New folder\\DD\\diffusion_bbbp\\test_pos.csv'

FORCE_RETRAIN = False
FORCE_RERUN_FEATURE_REDUCTION = False
FORCE_RERUN_TOP17_RESAMPLING = False

In [4]:
# Load headerless toxicity sequence files and assign binary labels.
def load_sequence_file(path: str, label: int) -> pd.DataFrame:
    df = pd.read_csv(path, header=None, names=['seq'])
    df = df[df['seq'].notna()].copy()
    df['seq'] = df['seq'].astype(str).str.strip().str.upper()
    df = df[df['seq'] != ''].reset_index(drop=True)
    df['label'] = int(label)
    return df

train_neg_df = load_sequence_file(TRAIN_NEG_PATH, label=0)
train_pos_df = load_sequence_file(TRAIN_POS_PATH, label=1)
test_neg_df = load_sequence_file(TEST_NEG_PATH, label=0)
test_pos_df = load_sequence_file(TEST_POS_PATH, label=1)

train_df = pd.concat([train_neg_df, train_pos_df], axis=0).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
test_df = pd.concat([test_neg_df, test_pos_df], axis=0).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

print('[Data] Train shape:', train_df.shape, '| label dist:', train_df['label'].value_counts().to_dict())
print('[Data] Test  shape:', test_df.shape, '| label dist:', test_df['label'].value_counts().to_dict())
print('[Data] Train preview:')
print(train_df.head(3))

def _pick_aaindex_features(target_n: int = 86) -> list[str]:
    selected = []
    for fid in aaindex1.record_codes():
        try:
            rec = aaindex1[fid]
            vals = np.asarray([float(rec.values[aa]) for aa in STANDARD_AA], dtype=float)
            if np.isfinite(vals).all():
                selected.append(fid)
        except Exception:
            continue
        if len(selected) >= target_n:
            break
    if len(selected) < target_n:
        raise ValueError(f'Only {len(selected)} AAIndex features found; expected {target_n}.')
    return selected

def build_aaindex_matrix(df: pd.DataFrame, feature_ids: list[str]) -> tuple[np.ndarray, np.ndarray, list[int]]:
    X_rows, y_rows, valid_rows = [], [], []
    for i, row in df.reset_index(drop=True).iterrows():
        residues = [aa for aa in str(row['seq']).upper() if aa in STANDARD_AA]
        if len(residues) == 0:
            continue

        row_vals = []
        bad_row = False
        for fid in feature_ids:
            try:
                rec = aaindex1[fid]
                row_vals.append(float(np.mean([float(rec.values[aa]) for aa in residues])))
            except Exception:
                bad_row = True
                break

        if bad_row:
            continue

        X_rows.append(row_vals)
        y_rows.append(int(row['label']))
        valid_rows.append(i)

    if len(X_rows) == 0:
        raise ValueError('No rows were encoded. Check sequence characters in inputs.')

    return np.asarray(X_rows, dtype=float), np.asarray(y_rows, dtype=int), valid_rows

feat_names = _pick_aaindex_features(target_n=86)
X_train_raw, y_train, train_valid_rows = build_aaindex_matrix(train_df, feat_names)
X_test_raw, y_test, test_valid_rows = build_aaindex_matrix(test_df, feat_names)

feature_cols = list(feat_names)

print('[AAIndex] Train encoded shape:', X_train_raw.shape, '| labels:', dict(zip(*np.unique(y_train, return_counts=True))))
print('[AAIndex] Test  encoded shape:', X_test_raw.shape, '| labels:', dict(zip(*np.unique(y_test, return_counts=True))))

train_encoded_df = pd.DataFrame(X_train_raw, columns=feat_names)
train_encoded_df['label'] = y_train
test_encoded_df = pd.DataFrame(X_test_raw, columns=feat_names)
test_encoded_df['label'] = y_test

train_encoded_df.to_csv('toxicity_train_aaindex_encoded.csv', index=False)
test_encoded_df.to_csv('toxicity_test_aaindex_encoded.csv', index=False)
print('[AAIndex] Saved -> toxicity_train_aaindex_encoded.csv')
print('[AAIndex] Saved -> toxicity_test_aaindex_encoded.csv')

[Data] Train shape: (8828, 2) | label dist: {0: 4414, 1: 4414}
[Data] Test  shape: (2208, 2) | label dist: {1: 1104, 0: 1104}
[Data] Train preview:
                        seq  label
0        VLLNSAAALVALDPGTGT      0
1  FLPLIASVAANLVPKIFCKITKKC      1
2            IDCSKVNLTAECSS      1
[AAIndex] Train encoded shape: (8828, 86) | labels: {np.int64(0): np.int64(4414), np.int64(1): np.int64(4414)}
[AAIndex] Test  encoded shape: (2208, 86) | labels: {np.int64(0): np.int64(1104), np.int64(1): np.int64(1104)}
[AAIndex] Saved -> toxicity_train_aaindex_encoded.csv
[AAIndex] Saved -> toxicity_test_aaindex_encoded.csv


In [5]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print('[Scale] X_train:', X_train.shape, '| X_test:', X_test.shape)

[Scale] X_train: (8828, 86) | X_test: (2208, 86)


In [6]:
from sklearn.model_selection import RandomizedSearchCV
import time

# Fast training controls
FAST_MODE = True
MAX_CV_SPLITS = 3 if FAST_MODE else 5
N_JOBS_CV = -1

def get_model_search_spaces() -> list[dict]:
    return [
        {
            'name': 'LogisticRegression',
            'estimator': LogisticRegression(max_iter=1000, solver='lbfgs', random_state=RANDOM_SEED),
            'search': 'grid',
            'param_grid': {'C': [0.1, 1, 10]},
        },
        {
            'name': 'RandomForest',
            'estimator': RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=1),
            'search': 'random',
            'param_distributions': {
                'n_estimators': [150, 250, 350],
                'max_depth': [5, 10, None],
                'min_samples_split': [5, 10, 20],
                'min_samples_leaf': [1, 2, 4],
                'max_features': ['sqrt', 'log2'],
                'class_weight': [None, 'balanced_subsample'],
            },
            'n_iter': 12 if FAST_MODE else 24,
        },
        {
            'name': 'XGBoost',
            'estimator': XGBClassifier(eval_metric='logloss', verbosity=0, random_state=RANDOM_SEED, n_jobs=1),
            'search': 'random',
            'param_distributions': {
                'learning_rate': [0.03, 0.05, 0.1],
                'n_estimators': [100, 150, 200],
                'max_depth': [3, 5, 7],
                'subsample': [0.8, 0.9, 1.0],
                'colsample_bytree': [0.8, 0.9, 1.0],
            },
            'n_iter': 12 if FAST_MODE else 24,
        },
        {
            'name': 'SVM',
            'estimator': SVC(probability=False, random_state=RANDOM_SEED),
            'search': 'grid',
            'param_grid': {
                'C': [0.1, 1, 10],
                'kernel': ['linear', 'rbf'],
                'gamma': ['scale'],
            },
        },
        {
            'name': 'KNN',
            'estimator': KNeighborsClassifier(n_jobs=1),
            'search': 'grid',
            'param_grid': {
                'n_neighbors': [5, 9, 15],
                'weights': ['uniform', 'distance'],
                'metric': ['euclidean', 'manhattan'],
            },
        },
    ]

def train_models(X_train: np.ndarray, y_train: np.ndarray, seed: int = RANDOM_SEED) -> tuple[dict, dict]:
    classes, counts = np.unique(y_train, return_counts=True)
    class_dist = {int(c): int(n) for c, n in zip(classes, counts)}
    if len(classes) < 2:
        raise ValueError(f'Training data must contain both classes. Found {class_dist}')

    n_splits = min(MAX_CV_SPLITS, int(counts.min()))
    if n_splits < 2:
        raise ValueError(f'Not enough minority samples for CV. Found {class_dist}')

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    trained, params = {}, {}

    print('[Train] y_train class distribution:', class_dist)
    print('[Train] Using StratifiedKFold n_splits =', n_splits)
    print('[Train] FAST_MODE =', FAST_MODE)

    total_start = time.time()

    for cfg in get_model_search_spaces():
        name = cfg['name']
        start = time.time()
        print(f'[Train] {name} search ...', end=' ')

        if cfg['search'] == 'random':
            search = RandomizedSearchCV(
                estimator=cfg['estimator'],
                param_distributions=cfg['param_distributions'],
                n_iter=cfg['n_iter'],
                cv=cv,
                scoring='roc_auc',
                refit=True,
                random_state=seed,
                n_jobs=N_JOBS_CV,
                verbose=0,
                pre_dispatch='2*n_jobs',
            )
        else:
            search = GridSearchCV(
                estimator=cfg['estimator'],
                param_grid=cfg['param_grid'],
                cv=cv,
                scoring='roc_auc',
                refit=True,
                n_jobs=N_JOBS_CV,
                verbose=0,
                pre_dispatch='2*n_jobs',
            )

        search.fit(X_train, y_train)
        trained[name] = search.best_estimator_
        params[name] = search.best_params_
        elapsed = time.time() - start
        print(f"[Done] {elapsed:.1f}s | Best params: {params[name]}")

    print(f"[Train] Total training time: {(time.time() - total_start)/60:.2f} minutes")
    return trained, params

if (not FORCE_RETRAIN) and ('trained_models' in globals()) and ('best_params' in globals()):
    print('[Train] Reusing cached trained models and parameters.')
else:
    trained_models, best_params = train_models(X_train, y_train)

[Train] y_train class distribution: {0: 4414, 1: 4414}
[Train] Using StratifiedKFold n_splits = 3
[Train] FAST_MODE = True
[Train] LogisticRegression search ... [Done] 13.9s | Best params: {'C': 0.1}
[Train] RandomForest search ... [Done] 96.5s | Best params: {'n_estimators': 350, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None, 'class_weight': None}
[Train] XGBoost search ... [Done] 36.7s | Best params: {'subsample': 0.9, 'n_estimators': 150, 'max_depth': 7, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
[Train] SVM search ... [Done] 63.9s | Best params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
[Train] KNN search ... [Done] 7.9s | Best params: {'metric': 'euclidean', 'n_neighbors': 15, 'weights': 'distance'}
[Train] Total training time: 3.65 minutes


In [7]:
def evaluate_models(trained: dict, X_test: np.ndarray, y_test: np.ndarray) -> pd.DataFrame:
    rows = []
    for name, model in trained.items():
        y_pred = model.predict(X_test)
        y_score = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_test)
        rows.append({
            'Model': name,
            'Accuracy': round(accuracy_score(y_test, y_pred), 4),
            'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
            'Recall': round(recall_score(y_test, y_pred, zero_division=0), 4),
            'F1': round(f1_score(y_test, y_pred, zero_division=0), 4),
            'MCC': round(matthews_corrcoef(y_test, y_pred), 4),
            'ROC-AUC': round(roc_auc_score(y_test, y_score), 4),
            'AUC-PR': round(average_precision_score(y_test, y_score), 4),
        })

    return pd.DataFrame(rows).sort_values(['ROC-AUC', 'AUC-PR'], ascending=False).reset_index(drop=True)

results_df = evaluate_models(trained_models, X_test, y_test)
print(results_df.to_string(index=False))
results_df.to_csv('toxicity_model_comparison.csv', index=False)
print('Saved -> toxicity_model_comparison.csv')

             Model  Accuracy  Precision  Recall     F1    MCC  ROC-AUC  AUC-PR
      RandomForest    0.8678     0.8988  0.8288 0.8624 0.7377   0.9347  0.9419
           XGBoost    0.8655     0.8861  0.8388 0.8618 0.7320   0.9311  0.9399
               KNN    0.8528     0.8671  0.8333 0.8499 0.7062   0.9259  0.9321
               SVM    0.8619     0.8703  0.8505 0.8603 0.7239   0.9246  0.9256
LogisticRegression    0.8111     0.8473  0.7591 0.8008 0.6257   0.8817  0.8981
Saved -> toxicity_model_comparison.csv


In [8]:
def select_best_model(results_df: pd.DataFrame, trained_models: dict, best_params: dict) -> tuple[str, object]:
    best_row = results_df.loc[results_df['ROC-AUC'].idxmax()]
    best_name = best_row['Model']
    best_est = trained_models[best_name]
    print('[Best] Model:', best_name)
    print('[Best] Params:', best_params[best_name])
    return best_name, best_est

best_name, best_model = select_best_model(results_df, trained_models, best_params)

[Best] Model: RandomForest
[Best] Params: {'n_estimators': 350, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None, 'class_weight': None}


In [9]:
def _svm_eval_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray) -> dict:
    return {
        'Accuracy': round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall': round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1': round(f1_score(y_true, y_pred, zero_division=0), 4),
        'MCC': round(matthews_corrcoef(y_true, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_true, y_score), 4),
        'AUC-PR': round(average_precision_score(y_true, y_score), 4),
    }

def run_fixed_svm_feature_reduction(
    X_train: np.ndarray, X_test: np.ndarray, y_train: np.ndarray, y_test: np.ndarray,
    feature_cols: list[str], aa_feature_names: list[str], fixed_params: dict,
    seed: int = RANDOM_SEED, top_k_start: int = 20, top_k_end: int = 5
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    aa_cols = [c for c in aa_feature_names if c in feature_cols]
    aa_idx_map = {c: i for i, c in enumerate(feature_cols)}
    aa_indices = [aa_idx_map[c] for c in aa_cols]

    X_train_aa = X_train[:, aa_indices]
    X_test_aa = X_test[:, aa_indices]

    svm_fixed = SVC(probability=True, random_state=seed, **fixed_params)
    svm_fixed.fit(X_train_aa, y_train)

    y_pred_86 = svm_fixed.predict(X_test_aa)
    y_score_86 = svm_fixed.predict_proba(X_test_aa)[:, 1]
    base_metrics = _svm_eval_metrics(y_test, y_pred_86, y_score_86)

    perm = permutation_importance(
        svm_fixed, X_test_aa, y_test, scoring='roc_auc', n_repeats=20, random_state=seed, n_jobs=-1
    )

    ranking_df = pd.DataFrame({
        'Feature': aa_cols,
        'ImportanceMean': perm.importances_mean,
        'ImportanceStd': perm.importances_std,
    }).sort_values('ImportanceMean', ascending=False).reset_index(drop=True)

    top20_df = ranking_df.head(top_k_start).copy()
    top20_features = top20_df['Feature'].tolist()

    rows = [{'NumFeatures': len(aa_cols), 'FeatureSet': 'all_aaindex_86', **base_metrics}]

    for k in range(top_k_start, top_k_end - 1, -1):
        selected = top20_features[:k]
        idx = [aa_cols.index(f) for f in selected]
        model_k = SVC(probability=True, random_state=seed, **fixed_params)
        model_k.fit(X_train_aa[:, idx], y_train)
        y_pred_k = model_k.predict(X_test_aa[:, idx])
        y_score_k = model_k.predict_proba(X_test_aa[:, idx])[:, 1]
        metrics_k = _svm_eval_metrics(y_test, y_pred_k, y_score_k)
        rows.append({'NumFeatures': k, 'FeatureSet': ';'.join(selected), **metrics_k})
        print(f"[6c] Top-{k:02d} ROC-AUC={metrics_k['ROC-AUC']:.4f} AUC-PR={metrics_k['AUC-PR']:.4f}")

    reduction_results_df = pd.DataFrame(rows).sort_values('NumFeatures', ascending=False).reset_index(drop=True)
    return ranking_df, top20_df, reduction_results_df

if (not FORCE_RERUN_FEATURE_REDUCTION) and ('svm_feature_ranking_df' in globals()) and ('svm_top20_df' in globals()) and ('svm_reduction_results_df' in globals()):
    print('[6c] Reusing cached feature reduction outputs.')
else:
    if 'SVM' not in best_params:
        raise ValueError('SVM best params not found. Run the training cell first.')

    fixed_svm_params = dict(best_params['SVM'])
    svm_feature_ranking_df, svm_top20_df, svm_reduction_results_df = run_fixed_svm_feature_reduction(
        X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test,
        feature_cols=feature_cols, aa_feature_names=list(feat_names), fixed_params=fixed_svm_params
    )

    print(svm_reduction_results_df[['NumFeatures', 'Accuracy', 'Precision', 'Recall', 'F1', 'MCC', 'ROC-AUC', 'AUC-PR']].to_string(index=False))
    svm_feature_ranking_df.to_csv('toxicity_svm_fixed_aaindex_feature_ranking.csv', index=False)
    svm_top20_df.to_csv('toxicity_svm_fixed_top20_features.csv', index=False)
    svm_reduction_results_df.to_csv('toxicity_svm_fixed_feature_reduction_20_to_5.csv', index=False)
    print('Saved -> toxicity_svm_fixed_aaindex_feature_ranking.csv')
    print('Saved -> toxicity_svm_fixed_top20_features.csv')
    print('Saved -> toxicity_svm_fixed_feature_reduction_20_to_5.csv')

[6c] Top-20 ROC-AUC=0.9245 AUC-PR=0.9234
[6c] Top-19 ROC-AUC=0.9213 AUC-PR=0.9179
[6c] Top-18 ROC-AUC=0.9215 AUC-PR=0.9182
[6c] Top-17 ROC-AUC=0.9202 AUC-PR=0.9186
[6c] Top-16 ROC-AUC=0.9190 AUC-PR=0.9198
[6c] Top-15 ROC-AUC=0.9155 AUC-PR=0.9165
[6c] Top-14 ROC-AUC=0.9146 AUC-PR=0.9151
[6c] Top-13 ROC-AUC=0.9115 AUC-PR=0.9142
[6c] Top-12 ROC-AUC=0.9112 AUC-PR=0.9140
[6c] Top-11 ROC-AUC=0.9099 AUC-PR=0.9156
[6c] Top-10 ROC-AUC=0.9066 AUC-PR=0.9123
[6c] Top-09 ROC-AUC=0.9020 AUC-PR=0.9090
[6c] Top-08 ROC-AUC=0.8988 AUC-PR=0.9051
[6c] Top-07 ROC-AUC=0.8899 AUC-PR=0.9048
[6c] Top-06 ROC-AUC=0.8802 AUC-PR=0.8929
[6c] Top-05 ROC-AUC=0.8711 AUC-PR=0.8844
 NumFeatures  Accuracy  Precision  Recall     F1    MCC  ROC-AUC  AUC-PR
          86    0.8619     0.8703  0.8505 0.8603 0.7239   0.9246  0.9256
          20    0.8578     0.8699  0.8415 0.8554 0.7160   0.9245  0.9234
          19    0.8646     0.8751  0.8505 0.8627 0.7295   0.9213  0.9179
          18    0.8628     0.8733  0.8487 0.8608 0.7

In [10]:
def run_fixed_svm_top17_resampling_comparison(
    X_train: np.ndarray, X_test: np.ndarray, y_train: np.ndarray, y_test: np.ndarray,
    feature_cols: list[str], aa_feature_names: list[str], ranking_df: pd.DataFrame,
    fixed_params: dict, target_k: int = 17, methods: list[str] | None = None,
    positive_label: int = 1, seed: int = RANDOM_SEED
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN

    methods = [m.lower() for m in (methods or ['random', 'smote', 'adasyn'])]
    aa_cols = [c for c in aa_feature_names if c in feature_cols]
    feature_cols_map = {c: i for i, c in enumerate(feature_cols)}
    aa_indices = [feature_cols_map[c] for c in aa_cols]

    X_train_aa = X_train[:, aa_indices]
    X_test_aa = X_test[:, aa_indices]

    topk_features = ranking_df['Feature'].head(target_k).tolist()
    aa_pos_map = {c: i for i, c in enumerate(aa_cols)}
    topk_idx = [aa_pos_map[f] for f in topk_features]

    X_train_k = X_train_aa[:, topk_idx]
    X_test_k = X_test_aa[:, topk_idx]

    classes, counts = np.unique(y_train, return_counts=True)
    class_count_map = {int(c): int(n) for c, n in zip(classes, counts)}
    minority_count = int(min(class_count_map.values()))

    sampler_map = {
        'random': RandomOverSampler(random_state=seed),
        'smote': SMOTE(random_state=seed, k_neighbors=min(5, minority_count - 1)),
        'adasyn': ADASYN(random_state=seed, n_neighbors=min(5, minority_count - 1)),
    }

    rows, method_meta = [], {}

    for method in methods:
        sampler = sampler_map[method]
        X_res, y_res = sampler.fit_resample(X_train_k, y_train)

        model = SVC(probability=True, random_state=seed, **fixed_params)
        model.fit(X_res, y_res)

        y_pred = model.predict(X_test_k)
        y_score = model.predict_proba(X_test_k)[:, 1]
        metrics = _svm_eval_metrics(y_test, y_pred, y_score)

        res_classes, res_counts = np.unique(y_res, return_counts=True)
        res_count_map = {int(c): int(n) for c, n in zip(res_classes, res_counts)}
        method_meta[method] = {'TrainClassCountBefore': class_count_map, 'TrainClassCountAfter': res_count_map}

        rows.append({
            'Method': method,
            'NumFeatures': target_k,
            'TrainPosBefore': int(class_count_map.get(positive_label, 0)),
            'TrainPosAfter': int(res_count_map.get(positive_label, 0)),
            'FixedParams': str(fixed_params),
            **metrics,
        })

        print(f"[6d] {method.upper()} {class_count_map} -> {res_count_map} | ROC-AUC={metrics['ROC-AUC']:.4f}")

    results_df = pd.DataFrame(rows).sort_values(['ROC-AUC', 'AUC-PR', 'MCC'], ascending=False).reset_index(drop=True)
    topk_df = pd.DataFrame({'Feature': topk_features})

    topk_df.to_csv('toxicity_svm_fixed_top17_features.csv', index=False)
    results_df.to_csv('toxicity_svm_fixed_top17_resampling_comparison.csv', index=False)
    print('Saved -> toxicity_svm_fixed_top17_features.csv')
    print('Saved -> toxicity_svm_fixed_top17_resampling_comparison.csv')

    return topk_df, results_df, method_meta

if 'fixed_svm_params' not in globals() and 'SVM' in best_params:
    fixed_svm_params = dict(best_params['SVM'])

if (not FORCE_RERUN_TOP17_RESAMPLING) and ('svm_top17_resampling_df' in globals()) and ('svm_top17_resampling_meta' in globals()):
    print('[6d] Reusing cached top-17 resampling outputs.')
else:
    svm_top17_df, svm_top17_resampling_df, svm_top17_resampling_meta = run_fixed_svm_top17_resampling_comparison(
        X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test,
        feature_cols=feature_cols, aa_feature_names=list(feat_names), ranking_df=svm_feature_ranking_df,
        fixed_params=fixed_svm_params, target_k=17, positive_label=1, methods=['random', 'smote', 'adasyn'], seed=RANDOM_SEED
    )

    print(svm_top17_resampling_df[['Method', 'NumFeatures', 'Accuracy', 'Precision', 'Recall', 'F1', 'MCC', 'ROC-AUC', 'AUC-PR']].to_string(index=False))

[6d] RANDOM {0: 4414, 1: 4414} -> {0: 4414, 1: 4414} | ROC-AUC=0.9202
[6d] SMOTE {0: 4414, 1: 4414} -> {0: 4414, 1: 4414} | ROC-AUC=0.9202
[6d] ADASYN {0: 4414, 1: 4414} -> {0: 4414, 1: 4414} | ROC-AUC=0.9202
Saved -> toxicity_svm_fixed_top17_features.csv
Saved -> toxicity_svm_fixed_top17_resampling_comparison.csv
Method  NumFeatures  Accuracy  Precision  Recall     F1    MCC  ROC-AUC  AUC-PR
random           17    0.8619     0.8758  0.8433 0.8593 0.7242   0.9202  0.9186
 smote           17    0.8619     0.8758  0.8433 0.8593 0.7242   0.9202  0.9186
adasyn           17    0.8619     0.8758  0.8433 0.8593 0.7242   0.9202  0.9186


In [11]:
print('=' * 70)
print('TOXICITY AAINDEX PIPELINE COMPLETE')
print('=' * 70)
print('[Summary] Train encoded shape:', X_train.shape)
print('[Summary] Test encoded shape :', X_test.shape)
print('[Summary] Best model        :', best_name)
print('[Summary] Best params       :', best_params[best_name])
print('\n[Summary] Model comparison:')
print(results_df.to_string(index=False))
print('\nSaved files:')
print(' - toxicity_train_aaindex_encoded.csv')
print(' - toxicity_test_aaindex_encoded.csv')
print(' - toxicity_model_comparison.csv')
print(' - toxicity_svm_fixed_aaindex_feature_ranking.csv')
print(' - toxicity_svm_fixed_top20_features.csv')
print(' - toxicity_svm_fixed_feature_reduction_20_to_5.csv')
print(' - toxicity_svm_fixed_top17_features.csv')
print(' - toxicity_svm_fixed_top17_resampling_comparison.csv')

TOXICITY AAINDEX PIPELINE COMPLETE
[Summary] Train encoded shape: (8828, 86)
[Summary] Test encoded shape : (2208, 86)
[Summary] Best model        : RandomForest
[Summary] Best params       : {'n_estimators': 350, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None, 'class_weight': None}

[Summary] Model comparison:
             Model  Accuracy  Precision  Recall     F1    MCC  ROC-AUC  AUC-PR
      RandomForest    0.8678     0.8988  0.8288 0.8624 0.7377   0.9347  0.9419
           XGBoost    0.8655     0.8861  0.8388 0.8618 0.7320   0.9311  0.9399
               KNN    0.8528     0.8671  0.8333 0.8499 0.7062   0.9259  0.9321
               SVM    0.8619     0.8703  0.8505 0.8603 0.7239   0.9246  0.9256
LogisticRegression    0.8111     0.8473  0.7591 0.8008 0.6257   0.8817  0.8981

Saved files:
 - toxicity_train_aaindex_encoded.csv
 - toxicity_test_aaindex_encoded.csv
 - toxicity_model_comparison.csv
 - toxicity_svm_fixed_aaindex_feature_ranking.csv